# NB03 · Thursday — Leakage Hunt
**Machine Learning in Finance · Week 3, Thursday**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB03_leakage_hunt.ipynb)

**Finance question:** A classmate's model predicts stock returns ten times better than anything we've seen. Should we trust it?

**Key idea:** **Data leakage**: a feature that secretly contains information from the future. Leaky models look brilliant in testing and fail the moment real money is involved.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## The story
A classmate sends you this notebook: *"I added one feature — a 12-month trend — and my model's R-squared
jumped way up. I think I've cracked the market!"* Your team's job: **find out what's going on**.
Run the cells, compare with your NB03 results, and find the line that uses information from the future.

In [ ]:
panel = mlfin.make_features()
prices = mlfin.load_prices("M")

# Classmate's extra feature: the stock's 12-month trend
trend = (prices / prices.shift(12) - 1).clip(-1, 3)   # cap extreme values
trend = trend.shift(-1)          # line the trend up with the panel's dates
long = trend.stack().rename("trend_12").reset_index()
long.columns = ["date", "ticker", "trend_12"]
panel = panel.merge(long, on=["date", "ticker"], how="left")
panel[["date", "ticker", "mom_12_1", "trend_12", "ret_next"]].head()

In [ ]:
from sklearn.linear_model import LinearRegression
FEATURES = ["mom_12_1", "rev_1", "size", "btm", "vol_12", "trend_12"]
train, test = mlfin.time_split(panel, "2017-12-31")
X_train, y_train, _ = mlfin.prepare_xy(train, FEATURES, "ret_next")
X_test, y_test, test_rows = mlfin.prepare_xy(test, FEATURES, "ret_next")
model = LinearRegression().fit(X_train, y_train)
print(f"R-squared, training: {model.score(X_train, y_train):.3f}")
print(f"R-squared, test:     {model.score(X_test, y_test):.3f}")

In [ ]:
mlfin.plot_deciles(model.predict(X_test), y_test, dates=test_rows["date"],
                   title="Classmate's model, test period")

## Your investigation
Work through these with your team (an AI assistant is allowed — but check what it says):
1. In NB03 the test R-squared was below 0.01. How many times bigger is this one? Is that believable for stock returns?
2. Look at `trend_12`. On a row dated **31 March**, which month's price does it use as its *latest* price?
   (Hint: what does `shift(-1)` do? Try `prices["AAPL"].tail(3)` and `prices["AAPL"].shift(-1).tail(3)`.)
3. Write one sentence explaining why this feature "knows" part of `ret_next`.
4. Fix the line in the cell below and re-run the model. What happens to R-squared?

In [ ]:
# scratch space for your investigation

In [ ]:
# 🟡 YOUR TURN — fix the feature, then re-run the two cells above with the fixed panel
SHIFT = -1   # ← the classmate's choice. What should it be so the trend uses only prices known at each date?

trend = (prices / prices.shift(12) - 1).clip(-1, 3).shift(SHIFT)
long = trend.stack().rename("trend_12").reset_index()
long.columns = ["date", "ticker", "trend_12"]
panel = panel.drop(columns="trend_12").merge(long, on=["date", "ticker"], how="left")

train, test = mlfin.time_split(panel, "2017-12-31")
X_train, y_train, _ = mlfin.prepare_xy(train, FEATURES, "ret_next")
X_test, y_test, _ = mlfin.prepare_xy(test, FEATURES, "ret_next")
model = LinearRegression().fit(X_train, y_train)
print(f"Fixed feature -> R-squared, test: {model.score(X_test, y_test):.4f}")

## Wrap-up
Leakage is the most common way smart people fool themselves with machine learning. Three habits that catch it:
* Ask of every feature: **"on this row's date, could I have known this?"**
* Be suspicious of results that are **far better than anything sensible** (stock-return R-squared is usually under 1%).
* Keep a **strict time order**: train on the past, test on the future.

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*